# NeoNatal Watch AI — Phase 6: CNN-LSTM Deep Learning

> **DEMO / SYNTHETIC DATA — NOT FOR CLINICAL USE**
>
> This notebook builds our first Deep Learning model. It learns directly from the 3D sliding windows `(time_steps, vital_signs)` rather than manually engineered flat features.

---

## Why CNN + LSTM?
- **CNN (1D Convolution)**: Rapidly scans the 30-minute window for sudden local patterns (e.g., a sharp oxygen drop).
- **LSTM (Long Short-Term Memory)**: Scans across time to capture long-term trends (e.g., gradual temperature changes).
- **No Manual Features**: It figures out the 'rolling means' and 'slopes' automatically during training.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import tensorflow as tf
from ml.models.cnn_lstm import build_cnn_lstm_model
from ml.evaluation.metrics import evaluate_model

print('Imports OK')
print(f"TensorFlow Version: {tf.__version__}")

In [ ]:
# Load 3D tensors generated in Phase 2
X_train = np.load('../data/processed/X_train.npy')
y_train = np.load('../data/processed/y_train.npy')
X_val = np.load('../data/processed/X_val.npy')
y_val = np.load('../data/processed/y_val.npy')
X_test = np.load('../data/processed/X_test.npy')
y_test = np.load('../data/processed/y_test.npy')

print(f"X_train shape: {X_train.shape} (Windows, Time-Steps, Features)")
print(f"y_train shape: {y_train.shape}")

In [ ]:
# Handling extreme class imbalance via weights
n_pos = y_train.sum()
n_neg = len(y_train) - n_pos
weight_for_1 = min(n_neg / n_pos, 50.0) if n_pos > 0 else 1.0

class_weight = {0: 1.0, 1: weight_for_1}
print(f"Class Weights: {class_weight}")

In [ ]:
# Build the Model Architecture
model = build_cnn_lstm_model(
    sequence_length=X_train.shape[1],
    n_features=X_train.shape[2],
    learning_rate=0.001
)

model.summary()

In [ ]:
# Train the Model (Reduced epochs for demonstration)
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_pr_auc", mode="max", patience=5, restore_best_weights=True
    )
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=5,
    batch_size=64,
    class_weight=class_weight,
    callbacks=callbacks
)

In [ ]:
# Evaluate on the Test Set
y_test_probs = model.predict(X_test).flatten()

metrics = evaluate_model(
    y_true=y_test,
    y_probs=y_test_probs,
    model_name="CNN_LSTM_Notebook",
    save_dir="../reports/figures"
)